# Lab 1: Telco Customer Churn Exploration

**Computing Intelligence**  
This notebook analyses the IBM Telco Customer Churn dataset. It loads the data, cleans it, converts the churn target from Yes/No to 1/0, checks data types and missing values, and explores how churn differs across demographic, service, and payment characteristics.

**Dataset:** [Kaggle: Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (IBM sample data).

In [ ]:
# Run this cell first. If the remote link is blocked on your network,
# download the Kaggle CSV and place it in data/Telco-Customer-Churn.csv.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
LOCAL_FILE = Path("data/Telco-Customer-Churn.csv")
LOCAL_FILE.parent.mkdir(exist_ok=True)
if LOCAL_FILE.exists():
    df = pd.read_csv(LOCAL_FILE)
else:
    df = pd.read_csv(DATA_URL)
    df.to_csv(LOCAL_FILE, index=False)
print(f"Dataset loaded: {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()

In [ ]:
# Last five records confirm that the complete file was loaded.
df.tail()

## Data preprocessing

The target is encoded numerically for analysis. `TotalCharges` is known to contain blank strings, so it is converted safely to a numeric value. After conversion, the missing-value check identifies any records that require removal.

In [ ]:
# Convert the churn target: Yes -> 1 and No -> 0.
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df[["customerID", "Churn"]].head()

In [ ]:
# Display original/current data types before final type adjustments.
df.dtypes.to_frame("Data type")

In [ ]:
# Convert TotalCharges to numeric. Blank values become NaN.
# Convert low-cardinality text attributes to category to represent their nature correctly.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
categorical_columns = df.select_dtypes(include="object").columns.drop("customerID")
df[categorical_columns] = df[categorical_columns].astype("category")
df["SeniorCitizen"] = df["SeniorCitizen"].astype("int64")
df["Churn"] = df["Churn"].astype("int64")
df.dtypes.to_frame("Data type after conversion")

In [ ]:
# Show only attributes that have null values.
null_counts = df.isnull().sum()
null_counts[null_counts > 0].to_frame("Null values")

In [ ]:
# Remove rows containing null values and verify that no nulls remain.
before_rows = len(df)
df = df.dropna().copy()
print(f"Removed {before_rows - len(df)} rows with missing values.")
print("Remaining null values:", int(df.isnull().sum().sum()))
print("Clean dataset shape:", df.shape)

## Exploration helper

Each chart reports both the number of customers and the within-group churn rate. The interpretation after every output explains what the result means.

In [ ]:
def churn_plot(column, title, order=None, rotate=0):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    sns.countplot(data=df, x=column, hue="Churn", order=order, ax=axes[0])
    axes[0].set_title(f"{title}: customer count by churn")
    axes[0].set_ylabel("Customers")
    axes[0].legend(title="Churn", labels=["No (0)", "Yes (1)"])
    axes[0].tick_params(axis="x", rotation=rotate)
    rates = df.groupby(column, observed=False)["Churn"].mean().mul(100).sort_values(ascending=False)
    sns.barplot(x=rates.index, y=rates.values, ax=axes[1], color="#4C72B0")
    axes[1].set_title(f"{title}: churn rate")
    axes[1].set_ylabel("Churn rate (%)")
    axes[1].set_xlabel(column)
    axes[1].tick_params(axis="x", rotation=rotate)
    for i, value in enumerate(rates.values):
        axes[1].text(i, value + 0.7, f"{value:.1f}%", ha="center", fontsize=9)
    plt.tight_layout()
    display(rates.to_frame("Churn rate (%)").round(1))
    plt.show()
    return rates

def interpretation(rates, highest_note=""):
    highest, lowest = rates.index[0], rates.index[-1]
    print(f"Interpretation: churn is highest for {highest} ({rates.iloc[0]:.1f}%) and lowest for {lowest} ({rates.iloc[-1]:.1f}%). {highest_note}")

### Demographic analysis

In [ ]:
rates = churn_plot("gender", "Gender distribution")
interpretation(rates, "The small difference suggests gender alone is not a strong churn driver.")

In [ ]:
rates = churn_plot("SeniorCitizen", "Senior-citizen status", order=[0, 1])
interpretation(rates, "Senior customers show a meaningfully higher churn rate, so age-related needs may deserve targeted retention support.")

In [ ]:
rates = churn_plot("Partner", "Partner status", order=["No", "Yes"])
interpretation(rates, "Customers without a partner tend to be less tied to the service and therefore more likely to leave.")

In [ ]:
rates = churn_plot("Dependents", "Dependents status", order=["No", "Yes"])
interpretation(rates, "Customers without dependents have substantially higher churn, indicating household stability is associated with retention.")

### Service analysis

In [ ]:
rates = churn_plot("PhoneService", "Phone service", order=["No", "Yes"])
interpretation(rates, "The rates are similar, so phone service by itself is not a decisive churn factor.")

### Payments and account analysis

In [ ]:
rates = churn_plot("Contract", "Contract type", order=["Month-to-month", "One year", "Two year"])
interpretation(rates, "Month-to-month customers are the clearest retention risk; longer contracts are strongly associated with lower churn.")

In [ ]:
rates = churn_plot("PaperlessBilling", "Paperless billing", order=["No", "Yes"])
interpretation(rates, "Paperless-billing customers churn more often in this dataset; this may reflect a customer segment rather than a causal effect.")

In [ ]:
rates = churn_plot("PaymentMethod", "Payment method", rotate=20)
interpretation(rates, "Electronic check users have the highest churn rate, whereas automatic payment methods are associated with better retention.")

## Conclusion

The strongest descriptive retention signal is contract type: month-to-month customers churn far more than customers with one- or two-year contracts. Customers without dependents, senior citizens, paperless-billing users, and electronic-check users also warrant attention. These observations are associations, not proof of causation, and should guide follow-up modelling or controlled retention experiments.